## Exercise — Fitting a straight line to the n-alcohols

### Task 1 — What least squares minimises

Start by copying this setup from the video into your solution block and running it. It builds the ten n-alcohols, computes their molecular weights and splits the table into training and test sets.

```py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from rdkit import Chem
from rdkit.Chem import Descriptors, Draw
from rdkit.Chem.Draw import IPythonConsole
df = pd.DataFrame({
    "name": ["methanol", "ethanol", "1-propanol", "1-butanol", "1-pentanol",
             "1-hexanol", "1-heptanol", "1-octanol", "1-nonanol", "1-decanol"],
    "smiles": ["CO", "CCO", "CCCO", "CCCCO", "CCCCCO",
               "CCCCCCO", "CCCCCCCO", "CCCCCCCCO", "CCCCCCCCCO", "CCCCCCCCCCO"],
    "bp_c": [64.7, 78.4, 97.2, 117.7, 138.0, 157.0, 176.0, 195.0, 214.0, 231.0],
})
mols = [Chem.MolFromSmiles(s) for s in df["smiles"]]
print("failed parses:", sum(1 for m in mols if m is None))
df["mw"] = [Descriptors.MolWt(m) for m in mols]
train = df.sample(frac=0.8, random_state=42)
test = df.drop(train.index)
```

Least squares picks the slope $m$ and intercept $b$ that make this quantity as small as possible:

$$\text{SSR} = \sum_{i=1}^{n} \left( y_i - (m\,x_i + b) \right)^2$$

Here are two candidate lines for boiling point against molecular weight:

| line | slope $m$ (°C per g/mol) | intercept $b$ (°C) |
|------|--------------------------|--------------------|
| A    | 1.3                      | 22                 |
| B    | 1.5                      | 0                  |

Write down which line you expect least squares to prefer. Then compute the SSR of each line over the **training** molecules. Use a generator expression with `zip` over `train["mw"]` and `train["bp_c"]`. Print both values, rounded to one decimal place.


### Task 2 — Fit the line with `curve_fit`

Here `curve_fit` searches for the slope and intercept with the smallest SSR, so you do not have to try candidates by hand. It is the same tool you used to fit a decay curve in track 1 video 12.

- Write a function `line(x, slope, intercept)` that returns slope times `x` plus intercept. The descriptor comes first and the parameters after it.
- Fit it with `curve_fit` to the training molecular weights and the training boiling points. Use the training set only.
- Store the result as `popt, pcov` and print `popt`.


### Task 3 — Read the parameters as chemistry

Unpack `popt` into two named variables, `slope` and `intercept`. Print the slope rounded to 3 decimals and the intercept rounded to 1 decimal. Put the unit of each in its label: °C per g/mol for the slope and °C for the intercept.

One CH2 group weighs 14.03 g/mol. Before you run anything, use the rule of thumb from the video to write down roughly how much the boiling point should rise per extra CH2. Then compute `slope * 14.03` and print it, rounded to 1 decimal, labelled as the boiling point rise per CH2 in °C.


### Task 4 — Predict the test molecules and plot the fit

Two lines are offered for getting predicted boiling points for the test molecules. Choose the one that **predicts** with the model you already fitted:

```py
test_pred = line(test["mw"], slope, intercept)
```

```py
popt_test, pcov_test = curve_fit(line, test["mw"], test["bp_c"])
```

Use your choice, then print a small table with the columns `name`, `measured_c` and `predicted_c`, rounded to 1 decimal.

Then draw one figure:
- the training molecules and the test molecules as two `plt.scatter` series, labelled `"training"` and `"test"`
- the fitted line drawn with `plt.plot` over 100 evenly spaced molecular weights from the smallest to the largest in `df`, labelled `"fitted line"`
- axis labels with units, a title, a legend and a grid, as in track 2 video 8

Save the figure as `alcohol_fit.png` with `dpi=300` before you show it.


### Task 5 — Where the line stops speaking

The line was learned from linear mono-alcohols between about 32 and 158 g/mol. Here are two molecules it has never seen:

| name            | SMILES          | measured bp (°C) |
|-----------------|-----------------|------------------|
| ethylene glycol | `OCCO`          | 197.3            |
| 1-dodecanol     | `CCCCCCCCCCCCO` | 259.0            |

Before running anything, write down two things for each molecule. First, is it in the training family, and is its weight inside the training range? Second, which of the two predictions do you expect to be badly wrong, and why?

Then run the check:
- parse both SMILES and print the number of failed parses
- print both molecular weights in g/mol, rounded to 1 decimal
- for each molecule, print the measured value and the value the line predicts, rounded to 1 decimal
- draw both molecules in a grid, with their names as legends
